# 99. 실습 인프라 지우기

`00_setup_infrastructure.ipynb`와 각 데모가 만든 리소스를 지웁니다. 데모 노트북의 정리 셀을 못 돌렸어도 여기서 함께 지웁니다.

- 데모가 만드는 개별 리소스: Demo 1 네트워크 실험(VPC·엔드포인트·Memory·Lambda), CloudWatch 경보·데이터 보호 정책, Demo 2 사용자 지정 평가기, Demo 3 가드레일·IAM 역할
- 공용 인프라: Runtime, Lambda·EventBridge, Knowledge Base·S3 Vectors·S3, Cognito, Secrets Manager, DynamoDB, 로그 그룹
- **Transaction Search**: 00에서 새로 켠 경우에만, `REVERT_TRANSACTION_SEARCH = True`일 때 되돌립니다.

Demo 1 VPC는 Lambda 네트워크 인터페이스가 풀릴 때까지 기다리므로 최대 20분 걸릴 수 있습니다.

In [ ]:
import os, boto3
os.environ.setdefault("LAB_SUFFIX", "demo")   # 리전은 00_setup이 저장한 설정 파일을 따름
import lab_infra as L, lab_network as N
REVERT_TRANSACTION_SEARCH = False
c = L.load(); R = L.REGION
print("설정 파일:", "있음" if c else "없음", "| 접두사:", L.P)

In [ ]:
out = []
# Demo 1
out += N.cleanup_network_lab()
cw = boto3.client("cloudwatch", region_name=R)
alarms = [a["AlarmName"] for a in cw.describe_alarms(AlarmNamePrefix=L.P)["MetricAlarms"]]
if alarms:
    out.append(L._try(f"CloudWatch 경보 {len(alarms)}개", cw.delete_alarms, AlarmNames=alarms))
if c.get("runtime_log_group"):
    out.append(L._try("로그 데이터 보호 정책", boto3.client("logs", region_name=R).delete_data_protection_policy,
                      logGroupIdentifier=c["runtime_log_group"]))
# Demo 2
ctl = boto3.client("bedrock-agentcore-control", region_name=R)
for e in ctl.list_evaluators(maxResults=100)["evaluators"]:
    if e.get("evaluatorName", "").startswith(L.PU):
        out.append(L._try(f"평가기 {e['evaluatorName']}", ctl.delete_evaluator, evaluatorId=e["evaluatorId"]))
# Demo 3
b = boto3.client("bedrock", region_name=R)
for g in b.list_guardrails(maxResults=100)["guardrails"]:
    if g["name"].startswith(L.P):
        out.append(L._try(f"가드레일 {g['name']}", b.delete_guardrail, guardrailIdentifier=g["id"]))
iam = boto3.client("iam")
for r in iam.list_roles(MaxItems=1000)["Roles"]:
    if r["RoleName"].startswith(L.P) and r["RoleName"].endswith("guardrail-enforced"):
        out.append(L._try(f"역할 {r['RoleName']}", L._delete_role, r["RoleName"]))
print("\n".join(out) or "데모 리소스 없음")

In [ ]:
print("\n".join(L.cleanup_all(revert_transaction_search=REVERT_TRANSACTION_SEARCH)))